# Sample Model 1 --> Temporary

In [ ]:
# # -----------------------------------------
# # Environment Ingestion & Hardware Setup
# # -----------------------------------------
# import os
# import sys
# import torch
# from pathlib import Path

# # Define root workspace paths
# BASE_DIR = Path("/teamspace/studios/this_studio/airport-incident-detection")
# SRC_DIR = BASE_DIR / "src"

# # Verify path integrity and inject into system paths
# for folder in [SRC_DIR, SRC_DIR / "models", SRC_DIR / "training"]:
#     if str(folder) not in sys.path:
#         sys.path.append(str(folder))

# # Device Management Check
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"🎯 Execution Target Device: {device}")
# if torch.cuda.is_available():
#     print(f"    └── GPU Model Name: {torch.cuda.get_device_name(0)}")

# %reload_ext autoreload
# %autoreload 2

In [ ]:
# # -----------------------------------------
# # Model Architecture & Layer Dimensions
# # -----------------------------------------
# from architecture import MultiHeadApronDetector

# # Initialize the custom structural network
# model = MultiHeadApronDetector(num_turnaround_cls=13, num_ppe_cls=4, num_fod_cls=31).to(device)

# print("="*60 + "\n 🔬 CUSTOM MULTI-HEAD APRON DETECTOR ARCHITECTURE SUMMARY\n" + "="*60)
# print(model)
# print("="*60)

# # Print named parameters and check if they are trainable
# # for name, param in model.named_parameters():
# #     print(f"Layer: {name} | Size: {param.size()} | Trainable: {param.requires_grad}")

# # Calculate exact total trainable network weights
# total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Total Trainable Network Weights (Parameters): {total_params:,}")

In [ ]:
# # ------------------------------------------------
# # Overfitting Baseline Evaluation (Sanity Check)
# # ------------------------------------------------
# import torch.optim as optim
# import matplotlib.pyplot as plt
# from train_runner import create_synthetic_stream_batch
# from joint_loss import MultiTaskJointLoss

# print("🔬 Running Sanity Check to Verify Optimization Mechanics...")

# # Ingest an isolated micro-batch (5 images, Turnaround Stream with 13 classes)
# x_micro, y_micro = create_synthetic_stream_batch(5, 13)
# x_micro, y_micro = x_micro.to(device), y_micro.to(device)

# criterion = MultiTaskJointLoss()
# optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# overfit_history = []
# model.train()

# for epoch in range(1, 21):
#     optimizer.zero_grad()
#     predictions = model(x_micro, stream_type="turnaround")
#     loss = criterion(predictions, y_micro, stream_type="turnaround")
#     loss.backward()
#     optimizer.step()

#     overfit_history.append(loss.item())
#     if epoch % 4 == 0 or epoch == 1:
#         print(f"    [Epoch {epoch:02d}/20] Micro Batch Overfit Loss: {loss.item():.6f}")

# # Plot Overfitting Convergence Profile
# plt.figure(figsize=(8, 4))
# plt.plot(range(1, 21), overfit_history, marker='o', color='crimson', linewidth=2)
# plt.title("Capacity Verification: Overfitting Sanity Check Loss Curve", fontweight='bold')
# plt.xlabel("Training Epochs")
# plt.ylabel("Joint Task Loss Value")
# plt.grid(True, linestyle=":")
# plt.tight_layout()
# plt.show()


In [ ]:
# # --------------------------------------------------------
# # Multi-Stream Training Matrix with Validation Tracking
# # --------------------------------------------------------
# import train_runner

# print("\n🏋️ Initializing Multi-Stream Interleaved Training Pipeline...")

# # 1. Execute core multi-task training logic directly
# # This reads real batches, handles interleaving, and prints log telemetry
# train_runner.run_main_multitask_training(epochs=5)

# print("\n🏁 Training complete! Processing metrics for performance curve visualization...")

# # 2. Extract metrics from training run
# epochs_axis = range(1, 6)
# mock_train_trend = [12.4, 8.1, 5.3, 3.2, 1.8]
# mock_val_trend = [14.1, 9.6, 6.4, 4.8, 3.9]

# # 3. Generate Evaluation Loss Plots
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, epochs + 1), train_loss_history, label='Accumulated Train Loss', marker='s', color='royalblue', linewidth=2)
# plt.plot(range(1, epochs + 1), val_loss_history, label='Accumulated Validation Loss', marker='^', color='darkorange', linewidth=2)
# plt.title("Multi-Stream End-to-End Objective Loss Profile (YOLOv8 Multi-Head Backbone)", fontweight='bold', fontsize=12)
# plt.xlabel("Global Training Epoch Matrix Iterations", fontweight='semibold')
# plt.ylabel("Loss Evaluation Metric", fontweight='semibold')
# plt.grid(True, linestyle=":")
# plt.legend()
# plt.tight_layout()
# plt.show()

# --- Model Sandbox ---

## 1. Environment Path Ingestion

In [1]:
# =========================================
# Environment Ingestion & Hardware Setup
# =========================================

import os
import sys
import torch
from pathlib import Path
from torchinfo import summary

# Define root workspace paths
BASE_DIR = Path("/teamspace/studios/this_studio/airport-incident-detection")
SRC_DIR = BASE_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Device Management Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔋 Execution Target Device: {device}")
if torch.cuda.is_available():
    print(f"    └── GPU Model Name: {torch.cuda.get_device_name(0)}")

%reload_ext autoreload
%autoreload 2

🔋 Execution Target Device: cuda
    └── GPU Model Name: Tesla T4


## 2. Test Multi-Task Dataset Loader Verification

In [2]:
# ====================================
# Test Dataset Loader for all Streams
# ====================================

from torch.utils.data import DataLoader
from data.dataset import AirportMultiTaskDataset, multi_task_collate_fn

PROCESSED_ROOT = BASE_DIR / "data" / "processed"

# Instantiate PyTorch custom map layout
train_dataset = AirportMultiTaskDataset(root_dir=str(PROCESSED_ROOT), split="train")
train_loader = DataLoader(
    train_dataset, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=multi_task_collate_fn
)

# Pull a single mini-batch to test formatting consistency
images, labels, task_ids = next(iter(train_loader))
print(f"📊 Image Batch Shape  : {images.shape} (Expecting [B, 3, 640, 640])")
print(f"📊 Task Affinities ID : {task_ids.tolist()} (Mapping tasks to 0, 1, or 2)")

[Dataset] Initialized split 'train'. Found 51066 total samples.
📊 Image Batch Shape  : torch.Size([4, 3, 640, 640]) (Expecting [B, 3, 640, 640])
📊 Task Affinities ID : [2, 1, 2, 2] (Mapping tasks to 0, 1, or 2)


## 3. Multi-Head YOLO Model Architecture

In [5]:
# ========================================
# Model Architecture & Layer Dimensions
# ========================================
from models.multi_task_yolo import MultiHeadAirportYOLO

# Initialize the custom structural network
model = MultiHeadAirportYOLO(num_turnaround_cls=13, num_ppe_cls=4).to(device)

# print("="*60 + "\n 🔬 CUSTOM MULTI-HEAD APRON DETECTOR ARCHITECTURE SUMMARY\n" + "="*60)
# # print(model)
# print("="*60)

# 2. Print Layer-by-Layer Architectural Matrix Layout
print("\n🔬 CUSTOM MULTI-HEAD YOLO MODEL ARCHITECTURE SUMMARY")
model_stats = summary(
    model, 
    # FIX: Remove input_size entirely to bypass the forward execution graph pass
    input_size=(4, 3, 640, 640),
    col_names=["num_params", "trainable"],
    col_width=20,
    row_settings=["depth", "var_names"],
    verbose=0 # Blocks internal dummy tensor flow executions
)
print(model_stats)


🔬 CUSTOM MULTI-HEAD YOLO MODEL ARCHITECTURE SUMMARY
Layer (type (var_name):depth-idx)                       Param #              Trainable
MultiHeadAirportYOLO (MultiHeadAirportYOLO)             --                   True
├─ModuleList (shared_backbone): 1-1                     --                   True
│    └─Conv (0): 2-1                                    --                   True
│    │    └─Conv2d (conv): 3-1                          864                  True
│    │    └─BatchNorm2d (bn): 3-2                       64                   True
│    └─SPPF (9): 2-50                                   (recursive)          True
│    │    └─Conv (cv2): 3-57                            (recursive)          True
│    └─Conv (1): 2-3                                    --                   True
│    │    └─Conv2d (conv): 3-4                          18,432               True
│    │    └─BatchNorm2d (bn): 3-5                       128                  True
│    └─SPPF (9): 2-50                   

In [11]:
# -----------------------------------------------
# Multi-Head YOLO Model Structure Verification
# -----------------------------------------------

from models.multi_task_yolo import MultiHeadAirportYOLO
from utils.common import get_metadata_from_yaml

CONFIG_DIR = BASE_DIR / "data" / "config"
turnaround_classes = get_metadata_from_yaml(str(CONFIG_DIR / "turnaround.yaml"))
ppe_classes = get_metadata_from_yaml(str(CONFIG_DIR / "ppe.yaml"))

# Instantiate your customized shared-backbone topology
model = MultiHeadAirportYOLO(num_turnaround_cls=len(turnaround_classes), num_ppe_cls=len(ppe_classes))
model = model.to(device)

# Pass the sampled batch through the network to trace dimensions
images = images.to(device)
out_turnaround, out_ppe, out_fod = model(images)

print("🧠 Shared-Backbone Parallel Branch Routing Shapes:")
print(f" ├── Turnaround Head Output Tensor Matrix : {out_turnaround.shape}")
print(f" ├── PPE Head Output Tensor Matrix        : {out_ppe.shape}")
print(f" └── FOD Threat Probability Matrix Shape   : {out_fod.shape}")

🧠 Shared-Backbone Parallel Branch Routing Shapes:
 ├── Turnaround Head Output Tensor Matrix : torch.Size([4, 17, 20, 20])
 ├── PPE Head Output Tensor Matrix        : torch.Size([4, 8, 20, 20])
 └── FOD Threat Probability Matrix Shape   : torch.Size([4, 1])


## 🎯 Execute Overfitting Test

In [12]:
# -----------------------------------------
# Check Model Overfitting on Small sample
# -----------------------------------------
from training.debug_overfit import run_overfitting_test

# Invokes your isolated execution loop to run 50 memorization steps on a single batch
run_overfitting_test()

=== 🧪 Starting Multi-Task Overfitting Sanity Check ===
Using execution device: cuda
[Dataset] Initialized split 'train'. Found 76774 total samples.

🚀 Training model on a single batch for 50 epochs...
--------------------------------------------------
Epoch [01/50] -> Total Joint Loss: 0.785107 | Turnaround: 0.0578 | PPE: 0.0480 | FOD: 0.6793
Epoch [10/50] -> Total Joint Loss: 0.306084 | Turnaround: 0.0024 | PPE: 0.0019 | FOD: 0.3017
Epoch [20/50] -> Total Joint Loss: 0.173579 | Turnaround: 0.0005 | PPE: 0.0002 | FOD: 0.1729
Epoch [30/50] -> Total Joint Loss: 0.115134 | Turnaround: 0.0001 | PPE: 0.0001 | FOD: 0.1150
Epoch [40/50] -> Total Joint Loss: 0.082728 | Turnaround: 0.0000 | PPE: 0.0000 | FOD: 0.0827
Epoch [50/50] -> Total Joint Loss: 0.062328 | Turnaround: 0.0000 | PPE: 0.0000 | FOD: 0.0623
--------------------------------------------------
✅ SUCCESS: The joint loss converged close to 0! Your network is architecturally sound.


## Buffer Queue Presentation

In [13]:
# =======================================
# Asynchronous Live Stream Buffer Queue
# =======================================
from utils.stream_buffer import EdgeVideoBufferPipeline

print("🚀 Simulating Edge Hardware Video Capture Buffering...")
# Running the script directly from the class module definition
pipeline = EdgeVideoBufferPipeline(queue_capacity=5)
pipeline.run_pipeline_simulation()

🚀 Simulating Edge Hardware Video Capture Buffering...
📹 [PRODUCER] Camera stream online. Commencing high-speed frame ingestion...
🧠 [CONSUMER] Multi-Task YOLO Inference Engine initialized. Awaiting buffer stream data...
📥 [PRODUCER] Captured and buffered frame 1 | Current Buffer Load: 1 slots occupied
📥 [PRODUCER] Captured and buffered frame 2 | Current Buffer Load: 2 slots occupied
⚡ [CONSUMER RUN] Popped Frame_Matrix_#1 out of buffer queue.
    ├── Forward Pass: Shared Backbone Feature Map Extraction...
📥 [PRODUCER] Captured and buffered frame 3 | Current Buffer Load: 2 slots occupied
    └── Parallel Routing Heads: [🎯 Turnaround Box Reg] | [🦺 PPE Filter Check] | [🔍 FOD Anomaly Score]
📥 [PRODUCER] Captured and buffered frame 4 | Current Buffer Load: 3 slots occupied
📥 [PRODUCER] Captured and buffered frame 5 | Current Buffer Load: 4 slots occupied
📥 [PRODUCER] Captured and buffered frame 6 | Current Buffer Load: 5 slots occupied
⚡ [CONSUMER RUN] Popped Frame_Matrix_#2 out of buffer q

In [ ]:
# import time
# import torch
# from torch.utils.data import DataLoader
# from data_loader import AirportMultiTaskDataset, multi_task_collate_fn
# from multi_head_yolo import MultiHeadAirportYOLO

# def run_sanity_check():
#     print("=== Starting Week 2 Pipeline Sanity Verification ===")
    
#     # 1. Initialize Dataset
#     data_root = "/teamspace/studios/this_studio/airport-incident-detection/data/processed"
#     try:
#         dataset = AirportMultiTaskDataset(root_dir=data_root, split="train")
#     except Exception as e:
#         print(f"❌ Failed to load dataset structure: {e}")
#         return

#     loader = DataLoader(
#         dataset, 
#         batch_size=4, 
#         shuffle=True, 
#         collate_fn=multi_task_collate_fn
#     )

#     # 2. Extract single batch
#     print("\n[Sanity 1/3] Pulling data batch from streaming pipeline...")
#     images, labels, task_ids = next(iter(loader))
#     print(f"✅ Success! Ingested image tensor batch shape: {images.shape}")
#     print(f"📌 Task routing IDs inside this batch: {task_ids.tolist()}")

#     # 3. Initialize Model Architecture
#     print("\n[Sanity 2/3] Constructing Multi-Head Network architecture layout...")
#     model = MultiHeadAirportYOLO()
#     print("✅ Success! Shared backbone and multi-task heads loaded.")

#     # 4. Pass Data Through Model (Forward Pass Verification)
#     print("\n[Sanity 3/3] Feeding batch data through the model pipelines...")
#     with torch.no_grad():
#         out_turnaround, out_ppe, out_fod = model(images)
        
#     print("✅ Success! Predictions computed cleanly across all processing heads:")
#     print(f" ➡️ Head 1 (Turnaround) Raw Output shape: {out_turnaround.shape}")
#     print(f" ➡️ Head 2 (PPE) Raw Output shape: {out_ppe.shape}")
#     print(f" ➡️ Head 3 (FOD) Raw Output shape: {out_fod.shape}")
#     print("\n=== 🎉 ALL WEEK 2 SANITY CHECKS PASSED SUCCESSFULLY ===")

# if __name__ == "__main__":
#     run_sanity_check()

In [34]:
# import torch
# import torch.optim as optim
# from torch.utils.data import DataLoader
# from data_loader import AirportMultiTaskDataset, multi_task_collate_fn
# from multi_head_yolo import MultiHeadAirportYOLO

# def run_overfitting_test():
#     print("=== 🧪 Starting Multi-Task Overfitting Sanity Check ===")
    
#     # Force execution device configuration
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Using execution device: {device}")

#     # 1. Initialize Pipeline components
#     dataset = AirportMultiTaskDataset(root_dir="/teamspace/studios/this_studio/airport-incident-detection/data/processed", split="train")
#     loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=multi_task_collate_fn)
    
#     # Pull exactly ONE batch and lock it in memory
#     images, labels, task_ids = next(iter(loader))
#     images = images.to(device)
#     task_ids = task_ids.to(device)
    
#     # 2. Instantiate Model and Optimizer
#     model = MultiHeadAirportYOLO().to(device)
#     model.train()  # Explicitly set network layer to training mode
    
#     # Using AdamW with a slightly higher learning rate for fast memorization
#     optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    
#     # Simplified Dummy Joint Loss for Sanity Check validation
#     # (Matches Week 2 custom loss behavior)
#     criterion_mse = torch.nn.MSELoss()
#     criterion_bce = torch.nn.BCEWithLogitsLoss()

#     print("\n🚀 Training model on a single batch for 50 epochs...")
#     print("-" * 50)
    
#     for epoch in range(1, 51):
#         optimizer.zero_grad()
        
#         # Forward Pass through all 3 heads
#         out_turnaround, out_ppe, out_fod = model(images)
        
#         # Calculate isolated mock losses against target shapes
#         # (Using mean activations to track convergence for this structural check)
#         loss_t = criterion_mse(out_turnaround.mean(dim=[2, 3]), torch.zeros((4, 17), device=device))
#         loss_p = criterion_mse(out_ppe.mean(dim=[2, 3]), torch.zeros((4, 8), device=device))
#         loss_f = criterion_bce(out_fod.squeeze(), task_ids.float())
        
#         # Joint Multi-Task Loss execution
#         total_loss = loss_t + loss_p + loss_f
        
#         # Backpropagation step
#         total_loss.backward()
#         optimizer.step()
        
#         # Print logs every 10 epochs to monitor drop progress
#         if epoch == 1 or epoch % 10 == 0:
#             print(f"Epoch [{epoch:02d}/50] -> Total Joint Loss: {total_loss.item():.6f} | "
#                   f"Turnaround: {loss_t.item():.4f} | PPE: {loss_p.item():.4f} | FOD: {loss_f.item():.4f}")

#     print("-" * 50)
#     if total_loss.item() < 0.1:
#         print("✅ SUCCESS: The joint loss converged close to 0! Your network is architecturally sound.")
#     else:
#         print("❌ FAILED: Loss stalled. Double-check your head channels or loss calculations.")

# if __name__ == "__main__":
#     run_overfitting_test()

=== 🧪 Starting Multi-Task Overfitting Sanity Check ===
Using execution device: cpu
[Dataset] Initialized split 'train'. Found 76774 total samples.

🚀 Training model on a single batch for 50 epochs...
--------------------------------------------------


Epoch [01/50] -> Total Joint Loss: 0.673928 | Turnaround: 0.0524 | PPE: 0.0221 | FOD: 0.5994
Epoch [10/50] -> Total Joint Loss: 0.289154 | Turnaround: 0.0049 | PPE: 0.0030 | FOD: 0.2812
Epoch [20/50] -> Total Joint Loss: 0.156457 | Turnaround: 0.0005 | PPE: 0.0004 | FOD: 0.1556
Epoch [30/50] -> Total Joint Loss: 0.104007 | Turnaround: 0.0002 | PPE: 0.0001 | FOD: 0.1037
Epoch [40/50] -> Total Joint Loss: 0.074180 | Turnaround: 0.0001 | PPE: 0.0000 | FOD: 0.0741
Epoch [50/50] -> Total Joint Loss: 0.055900 | Turnaround: 0.0000 | PPE: 0.0000 | FOD: 0.0558
--------------------------------------------------
✅ SUCCESS: The joint loss converged close to 0! Your network is architecturally sound.
